# Phase 1 · Evaluation on val and test
Scores models by retrieving from the frozen teacher corpus index built by `phase_1_teacher`. Single process, one GPU. Results are merged into `runs/<mode>/eval/metrics.json`.

**Corpus caveat:** the index holds only the positives + one stored negative of the core queries (~200k passages), not full MS MARCO (8.8M). Absolute numbers are therefore higher than published MS MARCO scores; use them to compare models against each other and against the teacher ceiling.

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import torch
from teacher import TEACHER_DEFAULTS as T

DEBUG = True    # evaluates the debug cache built by phase_1_teacher with DEBUG = True. Set False for the full run.

# --- paths: same cache as phase_1_teacher; results go to runs/<mode>/eval/metrics.json
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
DATA_PATH = f"{ROOT}/cache/{MODE}/data.json"
EMB_DIR = f"{ROOT}/cache/{MODE}/emb/teacher"
OUT_DIR = f"{ROOT}/runs/{MODE}/eval"

# --- evaluation (retrieval against the frozen teacher corpus index)
EVAL = dict(splits=("val", "test"),
            k=100,                 # top-k retrieved per query (Recall@100 needs 100)
            dim=T["dim"],          # MRL dim of the teacher index
            batch_size=256)        # query-encoding batch size
MRL_DIMS = (64, 128)               # extra teacher-ceiling rows at smaller dims (cached vectors truncated)

# --- models to score: name -> hub id or checkpoint dir. Add fine-tuned dirs here later, e.g.
#     "aligned": dict(model=f"{ROOT}/runs/{MODE}/align/final", query_prefix="", max_query_len=32, dtype="fp16")
MODELS = {
    "untrained_student": dict(model="Qwen/Qwen3-Embedding-0.6B", query_prefix="", max_query_len=32, dtype="fp16"),
}

# --- query latency on one GPU (the speed argument for the small student)
LATENCY = dict(run=True,
               models={"teacher_4B": dict(model=T["model"], query_prefix=T["query_prompt"],
                                          max_query_len=T["query_max_len"], dtype=T["dtype"]),
                       "student_0.6B": dict(model="Qwen/Qwen3-Embedding-0.6B", query_prefix="", max_query_len=32,
                                            dtype="fp16")},
               n_queries=2000, batch_sizes=(1, 8, 32, 128), repeats=3)

## 1 · Load and check the index
Teacher corpus + val/test query vectors, checked before anything is scored.

In [ ]:
from data import load_json
from evaluation import load_index
from model import get_device

device = get_device()
data = load_json(DATA_PATH)
docs, teacher_q = load_index(data, EMB_DIR, EVAL["splits"])   # asserts unit norms, no NaNs, ids + positives valid

## 2 · Teacher ceiling + MRL sweep
Teacher queries vs teacher corpus: the best a student can reach. The `teacher@d<dim>` rows truncate the cached vectors to show what smaller dims would cost.

In [ ]:
from evaluation import teacher_ceiling, save_metrics

rows = teacher_ceiling(data, docs, teacher_q, EVAL["k"], device, dims=MRL_DIMS)
save_metrics(OUT_DIR, rows)
for name, result in rows.items():
    print(name, {s: round(m["ndcg@10"], 4) for s, m in result.items()})

## 3 · Models
Each model embeds the val/test queries and retrieves from the teacher index. The untrained student is expected to score **near zero**: its embedding space is not the teacher's.

In [ ]:
from evaluation import evaluate_model, save_metrics

for name, m in MODELS.items():
    print(f"--- {name}: {m['model']}")
    result = evaluate_model(data=data, docs=docs, device=device, **EVAL, **m)
    save_metrics(OUT_DIR, {name: result})

## 4 · Summary
One table per split, teacher first. retention = nDCG@10 / teacher nDCG@10.

In [ ]:
from evaluation import summarize

for split in EVAL["splits"]:
    summarize(OUT_DIR, split)
    print()

## 5 · Query latency
ms/query and queries/s at several batch sizes on one GPU. Teacher queries include the instruction prompt, as they do in the index.

In [ ]:
from evaluation import measure_latency
from model import DTYPES, load_encoder
from teacher import free_gpu_memory

if LATENCY["run"]:
    texts = data["val"]["texts"][:LATENCY["n_queries"]]
    for name, m in LATENCY["models"].items():
        print(f"--- {name}: {m['model']}")
        model, tok = load_encoder(m["model"], device, DTYPES[m["dtype"]])
        measure_latency(model, tok, texts, device, EVAL["dim"], m["max_query_len"], m["query_prefix"],
                        LATENCY["batch_sizes"], LATENCY["repeats"])
        del model
        free_gpu_memory()